# Supply Chain Planning

**The problem.** A company makes 2 products and sells them in 9 markets. Goods move through a simple network:

- A **supplier** provides a limited amount of raw material (RM) for each product.
- **3 factories** turn the raw material into finished products.
- Products are shipped either **directly to a market** or **through one of 5 warehouses**. Factory-to-warehouse shipping has 2 transport modes.

**The decision.** How much of each product to ship along each route. This also sets how much each factory produces (everything it ships out) and how much demand goes unmet.

**The goal.** Maximise profit:

> sales revenue − raw material cost − production cost − transport cost − penalty for unmet demand

**The rules (constraints).**

1. **Raw material:** total production of each product, across all factories, cannot exceed the supplier's limit.
2. **Production capacity:** each factory has 2 operations shared by both products; its product mix must fit within each operation's capacity.
3. **Demand:** each market's demand for each product is met by shipments, and any gap is recorded as a shortage.
4. **Warehouse balance:** everything that goes into a warehouse must come out of it.
5. **Unavailable routes:** routes with a transport cost of 100 don't exist, so no flow is allowed on them.

All decisions are continuous, so this is a **linear program (LP)**. The notebook builds it, solves it with Gurobi, and reports the optimal plan, its sensitivity analysis (shadow prices, slacks, reduced costs), and the production at each factory.

In [ ]:
# install the third-party packages into this notebook's active Python environment
# gurobipy ships with a size-limited license, which is enough for this model
%pip install --quiet --disable-pip-version-check "gurobipy>=11" "pandas>=1.5"

In [ ]:
# import the Gurobi Python API and its constants (GRB.MAXIMIZE, GRB.OPTIMAL, ...)
import gurobipy as gp
from gurobipy import GRB

In [ ]:
# create the model; all decision variables are continuous,
# so Gurobi solves it as a linear program (LP)
SC = gp.Model("Supply_Chain_Planning")

In [ ]:
# products
P = [1,2]
# operations
O = [1,2]
# factories
F = [1,2,3]
# warehouses
W = [1,2,3,4,5]
# markets
M = [1,2,3,4,5,6,7,8,9]
# transport modes (factory to warehouse only)
MD = [1,2]

In [ ]:
# supplier RM capacity (product)
S = {1: 12, 2: 10.5}

In [ ]:
# shared production capacity of each operation (operation, product)
CAP = {(1,1): 5, (1,2): 5, (2,1): 8, (2,2): 4}

In [ ]:
# RM cost (product)
RC = {1: 4, 2: 3}

In [ ]:
# variable production cost (factory, product)
PC = {(1,1): 0.15, (1,2): 0.1, (2,1): 0.19, (2,2): 0.14, (3,1): 0.13, (3,2): 0.09}

In [ ]:
# transport cost - factory to market (factory, market)
TCFM = {(1,1): 100, (1,2): 0.6, (1,3): 100, (1,4): 100, (1,5): 100, (1,6): 100, (1,7): 100, (1,8): 100, (1,9): 100,
        (2,1): 100, (2,2): 100, (2,3): 100, (2,4): 100, (2,5): 100, (2,6): 100, (2,7): 100, (2,8): 100, (2,9): 100,
        (3,1): 100, (3,2): 100, (3,3): 100, (3,4): 100, (3,5): 100, (3,6): 100, (3,7): 0.5, (3,8): 0.6, (3,9): 100}

In [ ]:
# transport cost - factory to warehouse (factory, warehouse, transport mode)
TCFW = {(1,1,1): 1.1, (1,1,2): 100, (1,2,1): 0.7, (1,2,2): 0.5, (1,3,1): 0.6, (1,3,2): 100, (1,4,1): 0.7, (1,4,2): 100, (1,5,1): 0.9, (1,5,2): 0.7, 
       (2,1,1): 0.6, (2,1,2): 100, (2,2,1): 0.9, (2,2,2): 0.7, (2,3,1): 1.0, (2,3,2): 0.7, (2,4,1): 0.5, (2,4,2): 100, (2,5,1): 100, (2,5,2): 100,
       (3,1,1): 100, (3,1,2): 100, (3,2,1): 0.8, (3,2,2): 100, (3,3,1): 0.9, (3,3,2): 100, (3,4,1): 0.7, (3,4,2): 100, (3,5,1): 0.6, (3,5,2): 100}

In [ ]:
# transport cost - warehouse to market (warehouse, market)
TCWM = {(1,1): 0.1, (1,2): 0.2, (1,3): 0.15, (1,4): 0.12, (1,5): 0.18, (1,6): 0.1, (1,7): 100, (1,8): 100, (1,9): 100,
       (2,1): 0.15, (2,2): 0.12, (2,3): 0.1, (2,4): 0.17, (2,5): 100, (2,6): 100, (2,7): 100, (2,8): 100, (2,9): 100,
       (3,1): 0.18, (3,2): 0.15, (3,3): 100, (3,4): 0.15, (3,5): 100, (3,6): 100, (3,7): 100, (3,8): 100, (3,9): 100,
       (4,1): 100, (4,2): 100, (4,3): 100, (4,4): 0.14, (4,5): 0.13, (4,6): 100, (4,7): 0.18, (4,8): 0.1, (4,9): 0.14,
       (5,1): 100, (5,2): 100, (5,3): 100, (5,4): 100, (5,5): 100, (5,6): 0.19, (5,7): 0.14, (5,8): 0.16, (5,9): 0.12}

In [ ]:
# demand (market, product)
DEM = {(1,1): 0.750, (1,2): 1.000, (2,1): 1.300, (2,2): 0, (3,1): 0.850, (3,2): 1.450, (4,1): 0.750, (4,2): 1.250,
      (5,1): 0.550, (5,2): 0.950, (6,1): 0.200, (6,2): 0.600, (7,1): 0.700, (7,2): 1.500, (8,1): 0.450, (8,2): 1.500,
      (9,1): 0.750, (9,2): 0.900}

In [ ]:
# sales price (market, product)
SP = {(1,1): 6.5, (1,2): 5.5, (2,1): 7, (2,2): 5.1, (3,1): 6.3, (3,2): 5, (4,1): 7.6, (4,2): 5,
      (5,1): 7.2, (5,2): 5.4, (6,1): 6.5, (6,2): 5.2, (7,1): 7.9, (7,2): 5.7, (8,1): 7.2, (8,2): 5,
      (9,1): 7, (9,2): 5.2}

In [ ]:
# penalty cost per unit of unmet demand
COST_SHORT = 100
# transport cost that marks a lane as unavailable; flow on such lanes is fixed to 0
NO_LANE = 100

In [ ]:
# define variables - factory to market flows (product, factory, market)
FM = {(p,f,m): SC.addVar(lb=0.0, ub=0.0 if TCFM[f,m] >= NO_LANE else GRB.INFINITY, name=f"FM_{p}_{f}_{m}") for p in P for f in F for m in M}
SC.update()
FM

In [ ]:
# define variables - factory to warehouse flows (product, factory, warehouse, transport mode)
FW = {(p,f,w,md): SC.addVar(lb=0.0, ub=0.0 if TCFW[f,w,md] >= NO_LANE else GRB.INFINITY, name=f"FW_{p}_{f}_{w}_{md}") for p in P for f in F for w in W for md in MD}
SC.update()
FW

In [ ]:
# define variables - warehouse to market flows (product, warehouse, market)
WM = {(p,w,m): SC.addVar(lb=0.0, ub=0.0 if TCWM[w,m] >= NO_LANE else GRB.INFINITY, name=f"WM_{p}_{w}_{m}") for p in P for w in W for m in M}
SC.update()
WM

In [ ]:
# define variables - shortage (unmet demand) (market, product)
SHORT = {(m,p): SC.addVar(lb=0.0, ub=GRB.INFINITY, name=f"SHORT_{m}_{p}") for m in M for p in P}

In [ ]:
# add constraint: supplier RM limit
for p in P:
    SC.addConstr(gp.quicksum(FW[p,f,w,md] for f in F for w in W for md in MD) + gp.quicksum(FM[p,f,m] for f in F for m in M) <= S[p], name=f"RM_supplier_limit_{p}")

In [ ]:
# add constraint: production capacity
# CAP[o,p] is how much of product p operation o could make if it made only p.
# at each factory, the share of each operation's capacity used by all products must not exceed 100%
for f in F:
    for o in O:
        SC.addConstr(gp.quicksum((gp.quicksum(FW[p,f,w,md] for w in W for md in MD) + gp.quicksum(FM[p,f,m] for m in M))/CAP[o,p] for p in P) <= 1, name=f"production_capacity_{f}_{o}")

In [ ]:
# add constraint: demand is met by shipments, and any gap is recorded as shortage
for m in M:
    for p in P:
        SC.addConstr(gp.quicksum(FM[p,f,m] for f in F) + gp.quicksum(WM[p,w,m] for w in W) + SHORT[m,p] == DEM[m,p], name=f"demand_{m}_{p}")

In [ ]:
# add constraint: warehouse flow balance (inflow = outflow)
for w in W:
    for p in P:
        SC.addConstr(gp.quicksum(FW[p,f,w,md] for f in F for md in MD) == gp.quicksum(WM[p,w,m] for m in M), name=f"flow_balance_{w}_{p}")

In [ ]:
# add the objective function:
# profit = sales revenue - RM cost - production cost - transport cost (factory-market, factory-warehouse, warehouse-market) - shortage penalty
SC.setObjective(gp.quicksum(SP[m,p]*(gp.quicksum(FM[p,f,m] for f in F) + gp.quicksum(WM[p,w,m] for w in W)) for m in M for p in P) - gp.quicksum(RC[p]*(gp.quicksum(FW[p,f,w,md] for f in F for w in W for md in MD) + gp.quicksum(FM[p,f,m] for f in F for m in M)) for p in P) - gp.quicksum(PC[f,p]*(gp.quicksum(FW[p,f,w,md] for w in W for md in MD) + gp.quicksum(FM[p,f,m] for m in M)) for f in F for p in P) - gp.quicksum(TCFM[f,m]*FM[p,f,m] for p in P for f in F for m in M) - gp.quicksum(TCFW[f,w,md]*FW[p,f,w,md] for p in P for f in F for w in W for md in MD) - gp.quicksum(TCWM[w,m]*WM[p,w,m] for p in P for w in W for m in M) - gp.quicksum(COST_SHORT*SHORT[m,p] for m in M for p in P), GRB.MAXIMIZE)

In [ ]:
# solve the model and stop if no optimal solution was found
SC.optimize()
if SC.Status != GRB.OPTIMAL:
    raise RuntimeError(f"The solver did not find an optimal solution (status code: {SC.Status}).")
print("Solver status: OPTIMAL")

In [ ]:
# print every variable's value and the optimal profit
for v in SC.getVars():
    print(v.VarName, v.X)
print("Objective Fn Value =", SC.ObjVal)

In [ ]:
# constraint report: shadow price = change in profit per extra unit of the constraint's right-hand side;
# slack = unused room in the constraint (0 means the constraint is binding)
import pandas as pd
result_constraints = [{'name': c.ConstrName, 'shadow price': c.Pi, 'slack': c.Slack}
     for c in SC.getConstrs()]
print(pd.DataFrame(result_constraints))

In [ ]:
# variables used in the optimal plan (value >= 0.001), with their reduced costs
result_variables = [{'name': v.VarName, 'value': v.X, 'reduced cost': v.RC}
     for v in SC.getVars() if v.X >= 0.001]
print(pd.DataFrame(result_variables))

In [ ]:
# variables not used in the optimal plan (value < 0.001); reduced cost = how much profit
# would change per unit if the variable were forced into the plan
result_variables = [{'name': v.VarName, 'value': v.X, 'reduced cost': v.RC}
     for v in SC.getVars() if v.X < 0.001]
print(pd.DataFrame(result_variables))

In [ ]:
# total production of each product at each factory (direct shipments + shipments to warehouses)
for f in F:
    for p in P:
        X = sum(FM[p,f,m].X for m in M) + sum(FW[p,f,w,md].X for w in W for md in MD)
        print("Production at factory %g, product %g : " % (f,p), X)